# Downloading MODIS data from the NASA CMR

*Section 12 — Downloading data, notebook 4 of 8.* Previous: `03_nasa_cmr_download_gedi`. Next: `05_copernicus_odata_sen1`.

**You will learn:**

- how to query NASA's Common Metadata Repository (CMR) for product information and versions (MOD13Q1)
- how to search for data granules by bounding box and date with `rsgispy.dataaccess.nasa_cmr`
- how to build a download database and download the granules with your Earthdata login

**Before you start:** a NASA Earthdata Login account (https://urs.earthdata.nasa.gov). The searches do not need an account; the download does. Store the username and password in an encoded credentials file (`earthdata_userinfo.txt`, see *Username and password* below).

**Data:** downloaded from NASA Earthdata (via the CMR) into `outputs/`.

**Run time:** depends on the download; the search cells take about 1 minute.

> **Running this notebook:** every cell that logs in, queries the remote
> service or downloads data is tagged `skip-execution`, so the tutorial test
> runner does not run them. Run them yourself, in order, once you have an
> account and have created the credentials file.

## NASA Common Metadata Repository

[NASA's Common Metadata Repository (CMR)](https://www.earthdata.nasa.gov/eosdis/science-system-description/eosdis-components/cmr)
is a high-performance, high-quality, continuously evolving metadata system
that catalogs all data and service metadata records for NASA's Earth
Observing System Data and Information System (EOSDIS) and is the
authoritative management system for all EOSDIS metadata. These metadata
records are registered, modified, discovered, and accessed through
programmatic interfaces leveraging standard protocols and APIs.

You can search the CMR system online here: https://cmr.earthdata.nasa.gov/

In [1]:
import datetime
import os
import pprint

import rsgispy.dataaccess.nasa_cmr

In [2]:
# Output folder for the download database and the downloaded data.
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

# The encoded Earthdata username / password file (see below).
cred_file = "earthdata_userinfo.txt"

## Get information on a product

You can query the CMR for metadata on a product using the function below. You
can look up the available products and their names here:
https://earthdata.nasa.gov/eosdis/science-system-description/eosdis-standard-products

In [3]:
# MODIS NDVI Product
prod_name = "MOD13Q1"

In [ ]:
info_dict = rsgispy.dataaccess.nasa_cmr.get_prods_info(prod_short_name=prod_name)

In [ ]:
pprint.pprint(info_dict)

## Find most recent version

You can find the most recent version of the product you are interested in
using the function below:

In [ ]:
max_version = rsgispy.dataaccess.nasa_cmr.get_max_prod_version(
    prod_short_name=prod_name
)
max_version

## Search for data

Use the following function to search for the data you are interested in. You
can specify the bounding box (`[min_x, max_x, min_y, max_y]`, in degrees) but
also limit the search by date.

In [4]:
bbox = [39.1, 39.5, -8, -7.6]

In [ ]:
prod_data_granules = rsgispy.dataaccess.nasa_cmr.find_granules(
    prod_short_name=prod_name,
    version=max_version,
    only_dnwld=True,
    bbox=bbox,
    pt=None,
    start_date=datetime.datetime(year=2020, month=4, day=1),
    end_date=datetime.datetime(year=2020, month=4, day=30),
    cloud_min=0,
    cloud_max=None,
    sort_date=True,
    sort_desc=True,
    page_size=100,
    page_num=1,
    other_params=None,
)

In [ ]:
# Print out the number of data granules found
len(prod_data_granules)

### Print a single product to check the data type to download

Look up the file that you will want to download and note its data (MIME)
type, as you'll need it later. The MODIS HDF-EOS files are listed with the type `application/x-hdfeos`.

In [ ]:
pprint.pprint(prod_data_granules[0])

## How much data is to be downloaded?

The following function sums the file sizes of the identified data to download:

In [ ]:
total_file_size = rsgispy.dataaccess.nasa_cmr.get_total_file_size(
    granule_lst=prod_data_granules
)
total_file_size

## Get the download URLs

The next step is to build a JSON database of the data files to be downloaded.
The database records which files have been downloaded, so the download can be
re-run and will only fetch the files which are still missing.

*RSGISLib users:* RSGISLib used a `pysondb` database; rsgispy writes plain
JSON.

In [5]:
# The file name of the JSON download database
db_file_name = os.path.join(out_dir, f"{prod_name}_scns_db.json")

In [ ]:
urls_not_found = rsgispy.dataaccess.nasa_cmr.create_cmr_dwnld_db(
    db_json=db_file_name,
    granule_lst=prod_data_granules,
    dwnld_file_mime_type="application/x-hdfeos",
)

In [ ]:
# Check whether there were data for which the URL could not be found.
len(urls_not_found)

In [ ]:
urls_not_found

## Download the data

## Username and password

Never write your username and password into a notebook. rsgispy provides a
simple encoding of the username and password so that they are not stored as
plain text. Note that the encoding (base64) is **not** secure: keep the file
private and never commit it to version control.

Create the encoded file with the `rsgisuserpassfile` command-line tool, which
is installed with rsgispy and prompts for the password:

`rsgisuserpassfile -o earthdata_userinfo.txt -u <your NASA Earthdata username>`

or, from Python, with
`rsgispy.tools.credentials.create_username_password_file(username, password, out_file)`.
Once you have created `earthdata_userinfo.txt` (in this folder, or change the path
below), read it into your notebook/script with `get_username_password`.

The download function below reads the credentials file itself, so you only need to pass its path.

*RSGISLib users:* the tool was `rsgisuserpassfile.py <file>`; the output file
is now given with `-o`.

### Create output directory

In [6]:
out_dwnld_dir = os.path.join(out_dir, "MOD13Q1_data")
os.makedirs(out_dwnld_dir, exist_ok=True)

### Run download

In [ ]:
rsgispy.dataaccess.nasa_cmr.download_granules_use_dwnld_db(
    db_json=db_file_name,
    out_path=out_dwnld_dir,
    user_pass_file=cred_file,
    use_wget=False,
)